# 00 – Setup and access modes

The lab runs against one of four Iceberg REST catalogs (environment variable `CATALOG`), all with
the same object storage:

| Service | Role |
|---|---|
| **RustFS** (`rustfs:9000`, console `:9001`) | S3-compatible object storage; also has its own Iceberg catalog ("Table Buckets") |
| **Lakekeeper** (`lakekeeper:8181`) | Iceberg REST catalog with management API and UI (default) |
| **Apache Polaris** (`polaris:8181`) | Iceberg REST catalog with built-in roles |
| **Unity Catalog OSS** (`unitycatalog:8080`) | catalog with an Iceberg REST interface, volumes, functions, models |
| Trino, pg_lake, Flink | optional engines (compose profiles `trino`, `pglake`, `flink`) |

How engines get to the files depends on the catalog:

- *Vended credentials* (Lakekeeper, Polaris, Unity): the catalog fetches short-lived credentials restricted to the table through STS.
- *Remote signing* (Lakekeeper only, warehouse `demo-signing`): the catalog signs every single S3 request; engines never see credentials.
- RustFS vends no credentials: the engines use their own.


In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS, WAREHOUSES
from iceberg_lab.notebook import BUCKET, TRINO_SIGNING, TRINO_STS, catalog_limit, drop_table, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

## Setup (idempotent)
Sets up the catalog – different per catalog (Lakekeeper warehouses, Polaris catalog, RustFS table bucket, Unity catalog); also runs automatically when the tests start.


In [ ]:
from iceberg_lab import bootstrap

bootstrap.main()

## How the catalog is set up (each catalog has its own management API)


In [ ]:
from iceberg_lab.notebook import describe_catalog

describe_catalog()

## What does the catalog return when loading a table?

We create the same table in every warehouse of the catalog (Lakekeeper has two, the others
one) and compare the `config` the catalog returns for the table. Exactly this configuration decides how the engine accesses S3.

In [ ]:
import pyarrow as pa

from iceberg_lab.engines import pyiceberg_catalog

data = pa.table({"id": [1, 2, 3], "name": ["a", "b", "c"]})
configs = {}
for wh in WAREHOUSES:  # Lakekeeper: demo-signing and demo-sts; otherwise one warehouse
    cat = pyiceberg_catalog(wh)
    cat.create_namespace_if_not_exists("nb_setup")
    if cat.table_exists("nb_setup.demo"):
        drop_table(cat, "nb_setup.demo")
    t = cat.create_table("nb_setup.demo", schema=data.schema)
    t.append(data)
    configs[wh] = t.io.properties

def show(props):
    hidden = ("secret", "token")
    return {k: ("***" if any(h in k for h in hidden) else v)
            for k, v in sorted(props.items()) if k.startswith(("s3.", "py-io", "expiration"))}

for wh, props in configs.items():
    print(f"--- {wh}")
    for k, v in show(props).items():
        print(f"  {k} = {v}")

- Lakekeeper `demo-signing` returns `s3.signer.uri` / `s3.remote-signing-enabled` – PyIceberg switches to `FsspecFileIO` for that automatically.
- Warehouses with vended credentials return `s3.access-key-id`, `s3.secret-access-key`, `s3.session-token` and an expiry time. RustFS returns no credentials – the client's own keys are shown there.

## STS credentials are restricted to the table

In [ ]:
import boto3
from botocore.exceptions import ClientError

if not PROFILE.vended_credentials:
    print(f"Catalog {PROFILE.name} vends no credentials - the engines use their own")
else:
    p = configs[WAREHOUSE_STS]
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT, region_name=SETTINGS.s3_region,
                      aws_access_key_id=p["s3.access-key-id"],
                      aws_secret_access_key=p["s3.secret-access-key"],
                      aws_session_token=p["s3.session-token"])
    try:
        s3.list_objects_v2(Bucket=BUCKET, Prefix="")  # whole bucket
        print("Bucket listing allowed?!")
    except ClientError as e:
        print("Bucket listing refused:", e.response["Error"]["Code"])

## Look at the files in RustFS (with admin keys)

In [ ]:
admin = boto3.client("s3", endpoint_url=S3_ENDPOINT, region_name=SETTINGS.s3_region,
                     aws_access_key_id=SETTINGS.s3_access_key,
                     aws_secret_access_key=SETTINGS.s3_secret_key)
for obj in admin.list_objects_v2(Bucket=BUCKET, MaxKeys=15).get("Contents", []):
    print(f"{obj['Size']:>8}  {obj['Key']}")